In [ ]:
# ================================
# 1. IMPORTS
# ================================
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

# ================================
# 2. LOAD & PROCESS PDF
# ================================
def load_and_process_pdf(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)
    return chunks

# ================================
# 3. CREATE VECTOR STORE
# ================================
def create_vector_store(chunks):
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )
    return vectorstore

# ================================
# 4. DEFINE STATE
# ================================
class GraphState(TypedDict):
    query: str
    retrieved_docs: List[str]
    response: str
    confidence: float
    escalation: bool

# ================================
# 5. INITIALIZE LLM
# ================================
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# ================================
# 6. RETRIEVAL FUNCTION
# ================================
def retrieve_docs(state: GraphState):
    query = state["query"]
    docs = retriever.get_relevant_documents(query)

    return {
        "retrieved_docs": [doc.page_content for doc in docs]
    }

# ================================
# 7. GENERATE RESPONSE
# ================================
def generate_response(state: GraphState):
    context = "\n".join(state["retrieved_docs"])
    query = state["query"]

    prompt = f"""
    Answer the user's question using ONLY the context below.
    If you don't know, say "I don't know".

    Context:
    {context}

    Question:
    {query}
    """

    result = llm.invoke(prompt)
    answer = result.content

    # Simple confidence logic
    confidence = 0.9 if "I don't know" not in answer else 0.4

    return {
        "response": answer,
        "confidence": confidence
    }

# ================================
# 8. DECISION NODE
# ================================
def decision_node(state: GraphState):
    if state["confidence"] < 0.7 or len(state["retrieved_docs"]) == 0:
        return {"escalation": True}
    return {"escalation": False}

# ================================
# 9. HITL NODE
# ================================
def hitl_node(state: GraphState):
    print("\n⚠️ Escalation triggered. Human intervention required.")
    human_response = input("Enter human response: ")

    return {
        "response": human_response,
        "confidence": 1.0
    }

# ================================
# 10. OUTPUT NODE
# ================================
def output_node(state: GraphState):
    print("\n💬 Final Response:")
    print(state["response"])
    return state

# ================================
# 11. BUILD LANGGRAPH
# ================================
def build_graph():
    workflow = StateGraph(GraphState)

    workflow.add_node("retrieve", retrieve_docs)
    workflow.add_node("generate", generate_response)
    workflow.add_node("decision", decision_node)
    workflow.add_node("hitl", hitl_node)
    workflow.add_node("output", output_node)

    workflow.set_entry_point("retrieve")

    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", "decision")

    # Conditional routing
    workflow.add_conditional_edges(
        "decision",
        lambda state: "hitl" if state["escalation"] else "output",
        {
            "hitl": "hitl",
            "output": "output"
        }
    )

    workflow.add_edge("hitl", "output")
    workflow.add_edge("output", END)

    return workflow.compile()

# ================================
# 12. MAIN EXECUTION
# ================================
if __name__ == "__main__":
    # Load PDF
    chunks = load_and_process_pdf("sample.pdf")

    # Create Vector DB
    vectorstore = create_vector_store(chunks)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # Build Graph
    app = build_graph()

    print("\n🤖 RAG Customer Support Assistant Ready!\n")

    while True:
        query = input("\nAsk your question (or type 'exit'): ")
        if query.lower() == "exit":
            break

        initial_state = {
            "query": query,
            "retrieved_docs": [],
            "response": "",
            "confidence": 0.0,
            "escalation": False
        }

        app.invoke(initial_state)